In [2]:
import math
import warnings
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging

# Suppress warnings
warnings.filterwarnings("ignore")
logging.set_verbosity_error()

# Load GPT model
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set pad token to avoid warning
tokenizer.pad_token = tokenizer.eos_token

# Accept prompt from user
prompt = input("Enter your prompt: ")

# Encode input
inputs = tokenizer(prompt, return_tensors="pt")

# Generate response with token scores
output = model.generate(
    **inputs,
    max_new_tokens=20,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id,
    return_dict_in_generate=True,
    output_scores=True
)

# Display generated response
generated_tokens = output.sequences[0][inputs["input_ids"].shape[1]:]
generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("\nGenerated Response:")
print(generated_text)

print("\nToken Log Probabilities")
print("-" * 90)
print(f"{'Token':15} {'Log Probability':18} {'Probability':15} {'Cumulative Probability':24} {'Color'}")
print("-" * 90)

cumulative_probability = 1.0

for token_id, score in zip(generated_tokens, output.scores):
    probabilities = torch.softmax(score[0], dim=-1)

    probability = probabilities[token_id].item()
    log_probability = math.log(probability)
    cumulative_probability *= probability

    # Color coding
    if probability >= 0.50:
        color = "Green"
    elif probability >= 0.20:
        color = "Yellow"
    else:
        color = "Red"

    token = tokenizer.decode([token_id])

    print(f"{token:15} {log_probability:<18.6f} {probability:<15.6f} {cumulative_probability:<24.10f} {color}")


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Enter your prompt: Artificial Intelligence is

Generated Response:
 a tool to solve problems that are not solved by artificial intelligence (AI). It is a central part

Token Log Probabilities
------------------------------------------------------------------------------------------
Token           Log Probability    Probability     Cumulative Probability   Color
------------------------------------------------------------------------------------------
 a              -0.577046          0.561555        0.5615549088             Green
 tool           -3.098111          0.045134        0.0253454287             Red
 to             -2.418859          0.089023        0.0022563303             Red
 solve          -3.393477          0.033592        0.0000757939             Red
 problems       -0.488754          0.613390        0.0000464913             Green
 that           -1.247457          0.287234        0.0000133539             Yellow
 are            -0.881645          0.414101        0.0000